# Deep Learning for Physicists: A Physics-Education Review

**Jupyter reproducibility notebook - Paper 1, EJP review version**

This notebook accompanies the manuscript *Deep Learning for Physicists: A Physics-Education Review of Function Approximation and Scientific Generalization*. It reproduces the numerical workflow and supports two modes:

- **Frozen-data audit (default):** regenerates figures and tables from the released CSV/state files without retraining.
- **Full retraining:** reruns the neural models, multi-initialization checks, simple-pendulum regime-shift study, classical baselines, and figure generation.

The central validation hierarchy is:

$$\text{sample} \rightarrow \text{system} \rightarrow \text{parameter interpolation} \rightarrow \text{parameter extrapolation} \rightarrow \text{qualitative regime shift}.$$

The notebook does not claim a new architecture. It is designed so a student, instructor, or referee can audit exactly how the scientific validation question changes.

In [ ]:
# 1. Locate the project package
from pathlib import Path
import zipfile

ROOT = None
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates = [base] + [p for p in base.iterdir() if p.is_dir()]
    for candidate in candidates:
        if (candidate / "code" / "regenerate_plots_from_frozen_data.py").exists():
            ROOT = candidate
            break
    if ROOT is not None:
        break

if ROOT is None:
    # Optional local ZIP path for a notebook opened outside the unpacked
    # package.  Set this to the ZIP file before running the cell.
    PACKAGE_ZIP = Path('Deep_Learning_for_Physicists_Reproducibility.zip')
    if not PACKAGE_ZIP.exists():
        raise FileNotFoundError(
            "Unpack the project package before running this notebook, "
            "or set PACKAGE_ZIP to its local ZIP path."
        )
    extract_dir = Path.cwd() / 'deep_learning_package'
    extract_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(PACKAGE_ZIP) as zf:
        zf.extractall(extract_dir)
    candidates = list(extract_dir.glob('**/code/regenerate_plots_from_frozen_data.py'))
    if not candidates:
        raise RuntimeError("Could not locate the project code directory in the ZIP.")
    ROOT = candidates[0].parents[1]

print("Package root:", ROOT)
required = [ROOT/'requirements.txt', ROOT/'code', ROOT/'data', ROOT/'manuscript'/'figures']
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Incomplete package; missing: " + ", ".join(missing))
print("Package structure: OK")


In [ ]:
# 2. Install/check dependencies
import sys, subprocess
req = ROOT / 'requirements.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)], check=True)
print("Dependencies ready.")


In [ ]:
# 3. Choose reproducibility mode
FULL_RETRAIN = False

# FULL_RETRAIN=False is the fast audit path used to reproduce figures from frozen outputs.
# Set True to repeat the neural training and all benchmark stages.
print("Mode:", "FULL RETRAIN" if FULL_RETRAIN else "FROZEN-DATA AUDIT")


## Oscillator benchmark

The damped oscillator is

$$m\ddot{x}+c\dot{x}+kx=0,$$

parameterized by natural frequency $\omega_0$ and damping ratio $\zeta$. The same physical family is used to distinguish random point holdout from complete-system holdout, interpolation, extrapolation, and inverse parameter recovery.

In [ ]:
# 4. Run the oscillator workflow or audit frozen outputs
import subprocess, sys

def run(script):
    print(f"\n>>> {script}")
    subprocess.run([sys.executable, str(ROOT/'code'/script)], cwd=str(ROOT), check=True)

if FULL_RETRAIN:
    run('reproduce_workflow.py')
    run('multiseed_robustness.py')
    run('augment_baselines.py')
else:
    run('regenerate_plots_from_frozen_data.py')


In [ ]:
# 5. Inspect oscillator metrics
import pandas as pd, json
from IPython.display import display

with open(ROOT/'data'/'benchmark_summary.json') as f:
    summary = json.load(f)

keys = [
    'random_point_validation_mse',
    'random_split_unseen_trajectory_mse',
    'trajectory_split_unseen_trajectory_mse',
    'mean_interpolation_relative_l2',
    'mean_extrapolation_relative_l2',
    'true_omega0','estimated_omega0','true_zeta','estimated_zeta',
]
metrics = pd.DataFrame([(k, summary.get(k)) for k in keys], columns=['metric','value'])
display(metrics)

ms = pd.read_csv(ROOT/'data'/'multiseed_summary.csv')
print("\nThree-initialization robustness summary:")
display(ms)


In [ ]:
# 6. Display Figure 3 with external legend
from IPython.display import Image, display
p = ROOT/'plots'/'validation_protocol.png'
display(Image(filename=str(p)))


## Simple-pendulum regime shift

The second controlled system is an ideal simple pendulum,

$$\ddot{\theta}+\frac{g}{\ell}\sin\theta=0,$$

started from the bottom with angular speed $\omega_0$. The normalized energy separatrix is

$$E_s/\ell^2=2g/\ell,\qquad \omega_c=2\sqrt{g/\ell}.$$

Training systems are kept below $\omega_c$, where the pendulum executes bounded oscillations; held-out cases remain oscillatory, while regime-shift cases cross above $\omega_c$ into continuous rotation. This tests a qualitative physical change that is visible in a familiar undergraduate mechanics example.

In [ ]:
# 7. Run or audit the simple-pendulum benchmark
if FULL_RETRAIN:
    run('pendulum_regime.py')
    run('finalize_pendulum.py')
else:
    run('finalize_pendulum.py')

pendulum = pd.read_csv(ROOT/'data'/'pendulum_summary.csv')
display(pendulum)


In [ ]:
# 8. Display Figure 5
p = ROOT/'plots'/'pendulum_regime_shift.png'
display(Image(filename=str(p)))


## Scientific interpretation

A random test split answers whether the model predicts withheld samples drawn from a familiar collection of systems. It does **not** by itself establish performance on a new system, a new parameter region, or a new qualitative dynamical regime. The direct numerical/analytic model remains the reference for one-off forward solves; the surrogate is useful only when its amortized or differentiable role justifies the approximation.

## Teaching activities

The same notebook can be used as a computational-physics laboratory. Suggested activities are:

1. Replace the random point split with a trajectory-level split while keeping the network unchanged.
2. Compare interpolation and extrapolation errors against the training envelope.
3. Move the pendulum initial angular speed across the exact separatrix $\omega_c=2\sqrt{g/\ell}$ and explain the change from bounded oscillation to continuous rotation.
4. Reproduce the RBF and direct-solver baselines.
5. Use the differentiable surrogate to recover noisy oscillator parameters and compare with direct fitting.

For every activity, report what was withheld, which baseline was used, and which conclusions are empirical rather than universal.

In [ ]:
# 10. Create a compact teaching/audit manifest
manifest = {
    "manuscript": "Deep Learning for Physicists: A Physics-Education Review of Function Approximation and Scientific Generalization",
    "package_root": str(ROOT),
    "mode": "full_retrain" if FULL_RETRAIN else "frozen_data_audit",
    "learning_tasks": [
        "parameterized nonlinear approximation",
        "gradient-based optimization and automatic differentiation",
        "physically meaningful validation splits",
        "interpolation versus extrapolation",
        "qualitative regime-shift testing",
        "baseline-aware surrogate interpretation",
    ],
}
display(manifest)


In [ ]:
# 11. Display all manuscript benchmark figures
from PIL import Image as PILImage
import matplotlib.pyplot as plt

fig_names = [
    'oscillator_learning.png',
    'validation_protocol.png',
    'generalization_regimes.png',
    'pendulum_regime_shift.png',
    'baseline_comparison.png',
    'inverse_identification.png',
]
for name in fig_names:
    path = ROOT/'plots'/name
    if path.exists():
        display(Image(filename=str(path)))


In [ ]:
# 12. Export a compact audit bundle
import shutil
base_out = ROOT
out = base_out/'Deep_Learning_for_Physicists_Jupyter_Audit'
out.mkdir(parents=True, exist_ok=True)
for name in ['benchmark_summary.csv','multiseed_summary.csv','pendulum_summary.csv','rbf_generalization_summary.csv']:
    src = ROOT/'data'/name
    if src.exists(): shutil.copy2(src, out/name)
for name in ['validation_protocol.png','pendulum_regime_shift.png','baseline_comparison.png','inverse_identification.png']:
    src = ROOT/'plots'/name
    if src.exists(): shutil.copy2(src, out/name)
archive = shutil.make_archive(str(base_out/'Deep_Learning_for_Physicists_Jupyter_Audit'), 'zip', out)
print("Created:", archive)
print("The audit archive is available at the path above.")
